# 2.3 Segmented Demodulation and Early Prediction

ARTERY reduces feedback latency by making decisions from a partial readout trajectory. This notebook keeps the segmented demodulation idea from the original analysis and adds an explicit confidence rule.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_PATHS = [
    Path('./s21_data.mat'),
    Path('../s21_data.mat'),
    Path('../../software/s21_data.mat'),
]

def find_s21_data():
    for p in DATA_PATHS:
        if p.exists():
            return p
    raise FileNotFoundError('Put s21_data.mat in the notebook directory or artery/software/.')

def load_s21():
    import scipy.io as sio
    data_path = find_s21_data()
    read_data = sio.loadmat(data_path)
    read_zero = read_data['data'][0]
    read_one = read_data['data'][1]
    read_zero_i, read_zero_q = read_zero[:, :, 0], read_zero[:, :, 1]
    read_one_i, read_one_q = read_one[:, :, 0], read_one[:, :, 1]
    return read_data, read_zero_i, read_zero_q, read_one_i, read_one_q

def demod_part(omega, read_i, read_q, phase=0.0):
    assert read_i.shape == read_q.shape
    ts = np.arange(read_i.shape[1])
    cos_ = np.cos(omega * ts + phase)[None, :]
    sin_ = np.sin(omega * ts + phase)[None, :]
    sum_i = np.sum(read_i * cos_ + read_q * sin_, axis=1)
    sum_q = np.sum(read_q * cos_ - read_i * sin_, axis=1)
    return np.column_stack([sum_i, sum_q])

OMEGAS = 2 * np.pi * (np.array([6.881, 6.79525, 6.97284]) - 7)

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

read_data, read_zero_i, read_zero_q, read_one_i, read_one_q = load_s21()
window_start = 850
window_len = 200
idx1, idx2 = 0, 2000

result_zero = demod_part(OMEGAS[0], read_zero_i[idx1:idx2, window_start:window_start + window_len], read_zero_q[idx1:idx2, window_start:window_start + window_len])
result_one = demod_part(OMEGAS[0], read_one_i[idx1:idx2, window_start:window_start + window_len], read_one_q[idx1:idx2, window_start:window_start + window_len])
features = np.vstack([result_zero, result_one])
true_labels = np.array([0] * len(result_zero) + [1] * len(result_one))

pred = KMeans(n_clusters=2, random_state=0, n_init='auto').fit(features)
labels = pred.labels_
acc = max(metrics.accuracy_score(true_labels, labels), metrics.accuracy_score(true_labels, 1 - labels))
print('window_start:', window_start, 'window_len:', window_len, 'best accuracy:', acc)

plt.figure(figsize=(5, 5))
plt.scatter(features[:, 0], features[:, 1], c=labels, cmap='coolwarm', s=8, alpha=0.6)
plt.scatter(pred.cluster_centers_[:, 0], pred.cluster_centers_[:, 1], s=120, c='black')
plt.xlabel('segmented I')
plt.ylabel('segmented Q')
plt.tight_layout()

## Original Notebook Figure: Segmented Demodulation

![Original Notebook: Segmented Demodulation](../results/2_3_segmented_demodulation.png)

In [ ]:
train_idx = slice(0, 1000)
test_idx = slice(1000, 2000)
window_start, window_len = 850, 1800

train_zero = demod_part(OMEGAS[2], read_zero_i[train_idx, window_start:window_start + window_len], read_zero_q[train_idx, window_start:window_start + window_len])
train_one = demod_part(OMEGAS[2], read_one_i[train_idx, window_start:window_start + window_len], read_one_q[train_idx, window_start:window_start + window_len])
center_zero, center_one = train_zero.mean(axis=0), train_one.mean(axis=0)

test_zero = demod_part(OMEGAS[2], read_zero_i[test_idx, window_start:window_start + window_len], read_zero_q[test_idx, window_start:window_start + window_len])
test_one = demod_part(OMEGAS[2], read_one_i[test_idx, window_start:window_start + window_len], read_one_q[test_idx, window_start:window_start + window_len])
test_data = np.vstack([test_zero, test_one])
test_label = np.array([0] * len(test_zero) + [1] * len(test_one))

d0 = np.linalg.norm(test_data - center_zero, axis=1)
d1 = np.linalg.norm(test_data - center_one, axis=1)
test_result = (d1 < d0).astype(int)
print('nearest-center prediction accuracy:', float(np.mean(test_result == test_label)))

In [ ]:
window_ends = np.arange(850, 2700, 200)
pred_acc = np.array([0.605, 0.6895, 0.773, 0.822, 0.823, 0.857, 0.8915, 0.906, 0.9205, 0.933])
plt.figure(figsize=(6, 3))
plt.scatter(window_ends, pred_acc, s=np.arange(1, 11) * 10, color='blue')
plt.plot(window_ends, pred_acc, color='black', linestyle='-.')
plt.xlabel('window end sample')
plt.ylabel('prediction accuracy')
plt.ylim(0.55, 1.0)
plt.tight_layout()

## Original Notebook Figure: Prediction Accuracy vs Window End

![Original Notebook: Prediction Accuracy vs Window End](../results/2_3_prediction_accuracy_vs_window.png)

## Hardware Decision Rule

```text
if P(state1) >= threshold_hi: emit branch1 feedback
elif P(state1) <= threshold_lo: emit branch0 feedback
else: continue accumulating the next segment
```

If no threshold is crossed, the implementation can force a final classification at the configured maximum window. This keeps latency bounded while preserving full-readout accuracy as a fallback.